In [1]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import zipfile
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, roc_curve

sys.path.insert(0, "/kaggle/input/datasets/kabirsinghsachdeva/model-and-function")
from model import BaselineCNN
from losses import FocalLoss

# ---------------- config ----------------
DATASET       = "unbalanced"   # "unbalanced" = V16, "balanced" = output.zip from 02-preprocessing
IMAGE_FOLDERS = ["processed", "processed_bg"]   # one separate model is trained for each folder

# unbalanced (V16)
V16_ROOT = "/kaggle/input/datasets/nithyamajeti/v16-balanced-normalizedben-graham-and-non"
DR_SHARE = 0.40                # DR share of every training batch (13 of 32), the other 60% is No DR

# balanced (output.zip, images picked by DR_all.csv)
CSV_PATH    = "/kaggle/input/datasets/kabirsinghsachdeva/manifest-2balanced/DR_all.csv"
ZIP_PATH    = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip"
EXTRACT_DIR = "/kaggle/temp/balanced_images"   # /kaggle/temp is NOT saved into your version's output

# augmentation and 60:40 batches are only used when the data is unbalanced
USE_AUG       = DATASET == "unbalanced"
RATIO_BATCHES = DATASET == "unbalanced"

LOSS         = "focal"       # "bce" (control) or "focal"
IMG_SIZE     = 512
BATCH        = 32
EPOCHS       = 30
PATIENCE     = 5             # stop if val AUROC does not improve for this many epochs
LR, WD       = 1e-3, 1e-4
TARGET_SENS  = 0.90          # threshold is chosen on VAL to reach this sensitivity
SEED         = 42
SAMPLE_RUN   = False         # True = 10% of train + val, for debugging
OUT_DIR      = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device  = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device)
print("dataset:", DATASET, "| folders:", IMAGE_FOLDERS, "| loss:", LOSS,
      "| augmentation:", USE_AUG, "| 60:40 batches:", RATIO_BATCHES)

device: cuda
dataset: unbalanced | folders: ['processed', 'processed_bg'] | loss: focal | augmentation: True | 60:40 batches: True


In [2]:
!ls -d /kaggle/input/datasets/*/*/ && ls /kaggle/input/datasets/*/*v16*/

/kaggle/input/datasets/kabirsinghsachdeva/manifest-2balanced/
/kaggle/input/datasets/kabirsinghsachdeva/model-and-function/
/kaggle/input/datasets/nithyamajeti/v16-balanced-normalizedben-graham-and-non/
manifest_v2.csv  processed  processed_bg


In [3]:
ZIP_PATH = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip"

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
print(len(names))
print(names[:5])
print(sorted({n.split("/")[0] for n in names}))

177690
['__pycache__/', 'manifest_v2.csv', 'processed_bg/', 'processed_bg/val/', 'processed_bg/val/5398_left.jpg']
['__pycache__', 'manifest_v2.csv', 'processed', 'processed_bg']


In [4]:
def load_df(folder):
    """Manifest rows for one image folder, with the full image path in proc_path."""
    if DATASET == "unbalanced":
        df = pd.read_csv(V16_ROOT + "/manifest_v2.csv")
        df = df[df["split"].isin(["train", "val", "test"])]
        df = df[df["load_ok"].astype(str) == "True"].reset_index(drop=True)
        df["proc_path"] = V16_ROOT + "/" + folder + "/" + df["split"] + "/" + df["filename"]
        df.loc[~df["proc_path"].map(os.path.exists), "proc_path"] = None
    else:
        df = pd.read_csv(CSV_PATH)
        df = df[df["split"] != "unused"].reset_index(drop=True)
        folder_dir = os.path.join(EXTRACT_DIR, folder)
        # unzip only the images of this folder that are in DR_all.csv
        if not os.path.exists(folder_dir):
            needed = set(df["filename"])
            with zipfile.ZipFile(ZIP_PATH) as z:
                members = [n for n in z.namelist()
                           if n.startswith(folder + "/") and os.path.basename(n) in needed]
                print("extracting", len(members), "files from", folder)
                z.extractall(EXTRACT_DIR, members)
        index = {}
        for dirpath, _, files in os.walk(folder_dir):
            for f in files:
                index[f] = os.path.join(dirpath, f)
        df["proc_path"] = df["filename"].map(index)

    assert df["proc_path"].notna().all(), df[df["proc_path"].isna()].head()
    return df

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

eval_tfm = T.Compose([                                   # no random changes for val/test
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
if USE_AUG:
    # unbalanced: EVERY training image (No DR and DR) goes through the same random augmentation,
    # so repeated DR images always come out different and augmentation is not a hint for the label
    train_tfm = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomAffine(degrees=15, scale=(0.9, 1.1)),    # mild rotation and zoom
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
else:
    train_tfm = eval_tfm                                 # balanced: no augmentation

class DRDataset(Dataset):
    def __init__(self, frame, tfm):
        self.paths  = frame["proc_path"].tolist()
        self.labels = frame["label"].values.astype("float32")
        self.tfm = tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), torch.tensor(self.labels[i])

class RatioBatchSampler(Sampler):
    """Every batch is 60% No DR and 40% DR (19 + 13 for a batch of 32).
    No DR: shuffled once per epoch, each image used exactly once. The epoch ends when No DR runs out.
    DR: every DR image is used once first, then DR images are drawn at random to fill the rest.
    Repeats come out different because train_tfm is random every time an image is loaded."""
    def __init__(self, labels, batch_size, dr_share):
        self.neg = [i for i, y in enumerate(labels) if y == 0]
        self.pos = [i for i, y in enumerate(labels) if y == 1]
        self.n_dr  = round(batch_size * dr_share)
        self.n_neg = batch_size - self.n_dr
    def __len__(self):
        return (len(self.neg) + self.n_neg - 1) // self.n_neg
    def __iter__(self):
        neg = random.sample(self.neg, len(self.neg))     # new order every epoch
        pos = random.sample(self.pos, len(self.pos))
        for start in range(0, len(neg), self.n_neg):
            batch = neg[start:start + self.n_neg]
            k = round(len(batch) * self.n_dr / self.n_neg)   # keeps the ratio in the last, smaller batch
            for _ in range(k):
                batch.append(pos.pop() if pos else random.choice(self.pos))
            random.shuffle(batch)
            yield batch

def make_loaders(df):
    train_df = df[df["split"] == "train"]
    val_df   = df[df["split"] == "val"]
    test_df  = df[df["split"] == "test"]
    if SAMPLE_RUN:
        train_df = train_df.sample(frac=0.10, random_state=SEED)
        val_df   = val_df.sample(frac=0.10, random_state=SEED)
    print("train", len(train_df), "DR share", round(train_df["label"].mean(), 3))
    print("val  ", len(val_df),   "DR share", round(val_df["label"].mean(), 3))
    print("test ", len(test_df),  "DR share", round(test_df["label"].mean(), 3))

    train_ds = DRDataset(train_df, train_tfm)
    if RATIO_BATCHES:
        sampler = RatioBatchSampler(train_ds.labels, BATCH, DR_SHARE)
        print(f"each batch: {sampler.n_neg} No DR + {sampler.n_dr} DR | {len(sampler)} batches per epoch | "
              f"about {len(sampler) * sampler.n_dr / len(sampler.pos):.2f} uses per DR image per epoch")
        train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=4, pin_memory=use_amp)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                                  num_workers=4, pin_memory=use_amp, drop_last=True)
    val_loader  = DataLoader(DRDataset(val_df, eval_tfm), batch_size=BATCH, shuffle=False,
                             num_workers=4, pin_memory=use_amp)
    test_loader = DataLoader(DRDataset(test_df, eval_tfm), batch_size=BATCH, shuffle=False,
                             num_workers=4, pin_memory=use_amp)
    return train_loader, val_loader, test_loader

In [5]:
def get_loss(name):
    if name == "bce":
        return nn.BCEWithLogitsLoss()
    if name == "focal":
        return FocalLoss(gamma=2.0)
    raise ValueError(name)

In [6]:
def threshold_for_sensitivity(y, p, target):
    """Highest threshold whose sensitivity is at least target (this gives the best specificity)."""
    fpr, tpr, thr = roc_curve(y, p)
    i = int(np.argmax(tpr >= target))
    return float(thr[i]), float(tpr[i]), float(1 - fpr[i])

def evaluate(model, criterion, loader):
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                logits = model(x).squeeze(1)
            all_logits.append(logits.float().cpu()); all_y.append(y)
    logits = torch.cat(all_logits); y = torch.cat(all_y)
    probs = torch.sigmoid(logits).numpy()
    loss = criterion(logits, y).item()
    return y.numpy(), probs, loss

def train_one(folder, train_loader, val_loader):
    """Trains a fresh model on one image folder and saves the best checkpoint (by val AUROC)."""
    set_seed(SEED)
    model     = BaselineCNN().to(device)
    criterion = get_loss(LOSS)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scaler    = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, best_auc, bad_epochs = [], -1.0, 0
    ckpt = f"{OUT_DIR}/best_{LOSS}_{DATASET}_{folder}.pt"

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        run_loss, n = 0.0, 0                     # n = images actually seen this epoch
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device, enabled=use_amp):
                logits = model(x).squeeze(1)
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * len(y); n += len(y)

        yv, pv, val_loss = evaluate(model, criterion, val_loader)
        auc = roc_auc_score(yv, pv)
        thr, sens, spec = threshold_for_sensitivity(yv, pv, TARGET_SENS)
        history.append(dict(epoch=epoch, train_loss=run_loss / n, val_loss=val_loss,
                            val_auc=auc, val_sens_at_target=sens, val_spec_at_target=spec))
        print(f"[{folder}] epoch {epoch:2d} | train loss {run_loss / n:.4f} | val loss {val_loss:.4f} | "
              f"val AUROC {auc:.4f} | spec at {TARGET_SENS:.0%} sens {spec:.3f} | {time.time() - t0:.0f}s")

        if auc > best_auc:
            best_auc, bad_epochs = auc, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop"); break

    pd.DataFrame(history).to_csv(ckpt.replace(".pt", "_history.csv"), index=False)
    print(f"[{folder}] best val AUROC:", round(best_auc, 4), "| saved", ckpt)
    return model, criterion, ckpt, best_auc

In [7]:
def pick_threshold(folder, model, criterion, ckpt, best_auc, val_loader):
    """Loads the best checkpoint and freezes the threshold on VAL."""
    model.load_state_dict(torch.load(ckpt, map_location=device))
    yv, pv, _ = evaluate(model, criterion, val_loader)
    thr, sens, spec = threshold_for_sensitivity(yv, pv, TARGET_SENS)
    print(f"[{folder}] VAL: AUROC {roc_auc_score(yv, pv):.4f} | threshold {thr:.4f} -> sensitivity {sens:.3f}, specificity {spec:.3f}")
    json.dump(dict(loss=LOSS, dataset=DATASET, folder=folder, threshold=thr, seed=SEED,
                   img_size=IMG_SIZE, best_val_auc=float(best_auc)),
              open(ckpt.replace(".pt", "_threshold.json"), "w"))
    return thr

In [8]:
RUN_TEST = True          # set True only for the final run

def run_test(folder, model, criterion, ckpt, thr, test_loader):
    yt, pt, _ = evaluate(model, criterion, test_loader)
    pred = (pt >= thr).astype(int)
    tp = int(((pred == 1) & (yt == 1)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
    tn = int(((pred == 0) & (yt == 0)).sum()); fp = int(((pred == 1) & (yt == 0)).sum())
    res = dict(auroc=float(roc_auc_score(yt, pt)), sensitivity=tp / (tp + fn), specificity=tn / (tn + fp),
               precision=tp / max(tp + fp, 1), tp=tp, fn=fn, tn=tn, fp=fp, threshold=thr)
    print(f"[{folder}] TEST AUROC {res['auroc']:.4f}")
    print(f"sensitivity {res['sensitivity']:.3f} | specificity {res['specificity']:.3f} | precision {res['precision']:.3f}")
    print("confusion [[TN FP] [FN TP]]:", [[tn, fp], [fn, tp]])
    json.dump(res, open(ckpt.replace(".pt", "_test.json"), "w"))
    return res

# one separate model for each image folder (processed, processed_bg)
results = []
for folder in IMAGE_FOLDERS:
    print(f"\n==================== {LOSS} | {DATASET} | {folder} ====================")
    df = load_df(folder)
    train_loader, val_loader, test_loader = make_loaders(df)
    model, criterion, ckpt, best_auc = train_one(folder, train_loader, val_loader)
    thr = pick_threshold(folder, model, criterion, ckpt, best_auc, val_loader)

    row = dict(loss=LOSS, dataset=DATASET, folder=folder, best_val_auc=round(best_auc, 4), threshold=round(thr, 4))
    if RUN_TEST:
        res = run_test(folder, model, criterion, ckpt, thr, test_loader)
        for key in ["auroc", "sensitivity", "specificity", "precision"]:
            row["test_" + key] = round(res[key], 4)
    results.append(row)

    del model
    torch.cuda.empty_cache()

summary = pd.DataFrame(results)
summary.to_csv(f"{OUT_DIR}/summary_{LOSS}_{DATASET}.csv", index=False)
print()
print(summary.to_string(index=False))


==================== focal | unbalanced | processed ====================
train 62195 DR share 0.244
val   13326 DR share 0.247
test  13318 DR share 0.244
each batch: 19 No DR + 13 DR | 2476 batches per epoch | about 2.12 uses per DR image per epoch
[processed] epoch  1 | train loss 0.1679 | val loss 0.1542 | val AUROC 0.5993 | spec at 90% sens 0.170 | 1466s
[processed] epoch  2 | train loss 0.1665 | val loss 0.1558 | val AUROC 0.6038 | spec at 90% sens 0.165 | 666s
[processed] epoch  3 | train loss 0.1656 | val loss 0.1516 | val AUROC 0.6257 | spec at 90% sens 0.190 | 650s
[processed] epoch  4 | train loss 0.1651 | val loss 0.1478 | val AUROC 0.6312 | spec at 90% sens 0.193 | 631s
[processed] epoch  5 | train loss 0.1648 | val loss 0.1488 | val AUROC 0.6341 | spec at 90% sens 0.218 | 624s
[processed] epoch  6 | train loss 0.1639 | val loss 0.1493 | val AUROC 0.6484 | spec at 90% sens 0.223 | 623s
[processed] epoch  7 | train loss 0.1634 | val loss 0.1495 | val AUROC 0.6489 | spec at 9